<h2 style='border:0; color:#216969'>Table of Contents</h2>

- [**Notebook Imports**](#1)
- [**Importing Data**](#2)
- [**Feature Engineering**](#3)
- [**Exploratory Data Analysis**](#4)
- [**Transformation Pipeline**](#5)
- [**Random Forest**](#6)
- [**Hyperparameter Optimization**](#7)
- [**Final Predictions**](#8)

<div style='color: #216969;
           background-color: #EAF6F6;
           font-size: 200%;
           border-radius:15px;
           text-align:center;
           font-weight:600;
           border-style: solid;
           border-color: dark green;
           font-family: "Cambria";'>
Notebook Imports
<a class="anchor" id="1"></a> 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

import warnings
warnings.filterwarnings('ignore')

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

import optuna

import tensorflow as tf

<div style='color: #216969;
           background-color: #EAF6F6;
           font-size: 200%;
           border-radius:15px;
           text-align:center;
           font-weight:600;
           border-style: solid;
           border-color: dark green;
           font-family: "Cambria";'>
Importing data
<a class="anchor" id="2"></a> 

In [ ]:
data = pd.read_csv('../input/students-performance-in-exams/StudentsPerformance.csv')
data.head()

In [ ]:
data.isnull().sum()

- There are no missing values in this dataset

In [ ]:
data.duplicated().sum()

In [ ]:
data.describe()

> Mean maths score is lowest among the different type of scores. <br>
> Half of the students have marks greater than or equal to 70 in reading exam

In [ ]:
data.info()

<div style='color: #216969;
           background-color: #EAF6F6;
           font-size: 200%;
           border-radius:15px;
           text-align:center;
           font-weight:600;
           border-style: solid;
           border-color: dark green;
           font-family: "Cambria";'>
Feature Engineering
<a class="anchor" id="3"></a> 

- We can derive the following 2 features in this dataset using the different types of scores.
   - **Percentage**
   - **Grade**

In [ ]:
data['Percentage'] = round((data['reading score'] + data['writing score'] + data['math score']) / 3, 2)

In [ ]:
def Grade(percentage):
    if percentage >= 95: return "O"
    if percentage > 81 : return "A"
    if percentage > 71 : return "B"
    if percentage > 61 : return "C"
    if percentage > 51 : return "D"
    if percentage > 41 : return "E"
    else: return "F"

data["Grade"] = data['Percentage'].apply(lambda x: Grade(x))

In [ ]:
data.head()

<div style='color: #216969;
           background-color: #EAF6F6;
           font-size: 200%;
           border-radius:15px;
           text-align:center;
           font-weight:600;
           border-style: solid;
           border-color: dark green;
           font-family: "Cambria";'>
Exploratory Data Analysis
<a class="anchor" id="4"></a> 

In [ ]:
sns.set_context('notebook', font_scale= 1.3)
plt.rcParams['figure.facecolor'] = "#ffffe6"
plt.rcParams['axes.facecolor'] = "#ffffe6"
fig, ax = plt.subplots(1, 3, figsize = (20, 6))
ax1 = sns.violinplot(x = data['gender'], y = data['writing score'], palette= 'crest', ax= ax[0])
ax1 = sns.violinplot(x = data['gender'], y = data['reading score'], palette= 'crest', ax= ax[1])
ax1 = sns.violinplot(x = data['gender'], y = data['math score'], palette= 'crest', ax= ax[2])

In [ ]:
sns.set_context('notebook', font_scale= 1.3)
plt.rcParams['figure.facecolor'] = "#ffffe6"
plt.rcParams['axes.facecolor'] = "#ffffe6"
fig, ax = plt.subplots(1, 3, figsize = (20, 6))
ax1 = sns.histplot(x = data['writing score'], hue = data['gender'] , palette= 'plasma', ax= ax[0])
ax1 = sns.histplot(x = data['reading score'], hue = data['gender'], palette= 'plasma', ax= ax[1])
ax1 = sns.histplot(x = data['math score'], hue = data['gender'], palette= 'plasma', ax= ax[2])

In [ ]:
gender_mean_score = data.groupby('gender')[['math score', 'reading score', 'writing score']].mean().round(2).transpose()
fig = go.Figure(data = [
    go.Table(
        header = {
            'values': ['', '<b>Male</b>', '<b>Female</b>'],
            'line_color' : 'darkslategrey',
            'fill_color' : 'lightskyblue',
            'align' : 'center',
            'height' : 40,
            'font_size': 20
        },
        cells = {
            'values' : [gender_mean_score.index, gender_mean_score['male'], gender_mean_score['female']],
            'line_color' : 'darkslategrey',
            'fill_color' : 'lightcyan',
            'align' : 'center',
            'height' : 40,
            'font_size': 20
        }
    )
])

fig.update_layout(width = 600, height = 400)
fig.show()

- Females tend to do better than males in both reading and writing
- Males perform better in Maths

In [ ]:
plt.rcParams['figure.facecolor'] = "#ffffe6"
plt.rcParams['axes.facecolor'] = "#ffffe6"

grid = sns.PairGrid(data, vars=['math score', 'reading score', 'writing score', 'Percentage'], hue= 'gender', palette= 'hot_r',
                    height=2, aspect = 2)

grid = grid.map_diag(sns.histplot)
grid = grid.map_lower(sns.scatterplot, alpha =  0.7)
grid = grid.map_upper(sns.kdeplot, n_levels = 10, shade = True)

grid.add_legend()
plt.show()

In [ ]:
with sns.axes_style('white'):
    plt.figure(figsize= (20, 8))
    sns.heatmap(data.corr(), annot = True, fmt = '.2f', linewidths= 0.8, cmap="YlGnBu")

- Almost all of these scores are highly correlated with each other
- Maths score seems to be the least correlated among these, therefore we will try to predict maths score during modelling

In [ ]:
ethnicity = data['race/ethnicity'].value_counts()
fig = px.pie(values = ethnicity.values,
             names = ethnicity.index,
             color_discrete_sequence = px.colors.sequential.Sunset,
             title = "Race/Ethnicity distribution",
             hole = 0.8)
 
fig.update_traces(textinfo = 'label+percent', textfont_size=18)

fig.update_layout(
    font = dict(size = 20, family = "arial"),
    annotations = [dict(text = 'Race', x = 0.5, y = 0.5, font_size = 30, showarrow=False)]
)
fig.show()

In [ ]:
parental_education = data['parental level of education'].value_counts()
fig = px.pie(values = parental_education.values,
             names = parental_education.index,
             color_discrete_sequence = px.colors.sequential.Sunset,
             title = "Parental Level Of Education",
             hole = 0.8)
 
fig.update_traces(textinfo = 'label+percent', textfont_size=14)

fig.update_layout(
    font = dict(size = 15, family = "arial"),
    annotations = [dict(text = "Parent's Education", x = 0.5, y = 0.5, font_size = 20, showarrow=False)]
)
fig.show()

In [ ]:
plt.rcParams['figure.facecolor'] = "#ffffe6"
plt.rcParams['axes.facecolor'] = "#ffffe6"

fig, ax = plt.subplots(2, 3, figsize = (25, 12))
ax[1, 2].axis('off')

sns.kdeplot(data = data, x = "Percentage", hue = "gender", palette = 'inferno', cumulative = True, common_norm = False, ax = ax[0, 0])
sns.kdeplot(data = data, x = "Percentage", hue = "race/ethnicity", palette = 'inferno', cumulative = True, common_norm = False, ax = ax[0, 1])
sns.kdeplot(data = data, x = "Percentage", hue = "lunch", palette = 'inferno', cumulative = True, common_norm = False, ax = ax[0, 2])
sns.kdeplot(data = data, x = "Percentage", hue = "parental level of education", palette = 'inferno', cumulative = True, common_norm = False, ax = ax[1, 0])
sns.kdeplot(data = data, x = "Percentage", hue = "test preparation course", palette = 'inferno', cumulative = True, common_norm = False, ax = ax[1, 1])

plt.show()

- Females have higher percentage than males
- Students whose parents holds a master's degree have a higher percentage than others
- Students who completed their course have higher percentage as compared to those who didn't.

In [ ]:
plt.figure(figsize = (10, 6))
sns.barplot(x = 'Grade', y = 'Percentage', data= data, hue= 'gender', palette= 'crest');

In [ ]:
sns.set_palette("plasma")
data.groupby('parental level of education').agg('mean').sort_values(by = 'Percentage').plot(kind='barh',figsize=(15,10))
plt.legend(bbox_to_anchor=(1.03, 1), loc = 2);

- Students whose parents never went to college seems to have the lowest percentage
- Students whose parents have a master's degree performed the best followed by parents having a bachelor's degree

In [ ]:
sns.set_palette("crest")
data.groupby('race/ethnicity').agg('mean').sort_values(by = 'Percentage').plot(kind='barh',figsize=(15,10))
plt.legend(bbox_to_anchor=(1.03, 1), loc = 2);

<div style='color: #216969;
           background-color: #EAF6F6;
           font-size: 200%;
           border-radius:15px;
           text-align:center;
           font-weight:600;
           border-style: solid;
           border-color: dark green;
           font-family: "Cambria";'>
Transformation Pipeline
<a class="anchor" id="5"></a> 

In [ ]:
class CustomOrdinalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, grades_ordering = ['F', 'E', 'D', 'C', 'B', 'A', 'O'],
                 ethnicity_ordering = ['group A', 'group B', 'group C', 'group D', 'group E'],
                 parents_education_ordering = ['high school', 'some high school', 'some college', "associate's degree", "bachelor's degree", "master's degree"]):
                 
        self.grades_ordering = grades_ordering
        self.ethnicity_ordering = ethnicity_ordering
        self.parents_education_ordering = parents_education_ordering
    
    def fit(self, X, y=None):
        return self  

    def transform(self, X):
        X["Grade"] = X['Grade'].apply(lambda x: self.grades_ordering.index(x))
        X["parental level of education"] = X['parental level of education'].apply(lambda x: self.parents_education_ordering.index(x))
        X["race/ethnicity"] = X['race/ethnicity'].apply(lambda x: self.ethnicity_ordering.index(x))
        return X


In [ ]:
num_cols = ['reading score', 'writing score', 'Percentage']
cat_cols = ['gender', 'lunch', 'test preparation course']
ordinal_cols = ['Grade', 'race/ethnicity', 'parental level of education']


In [ ]:
pipeline = ColumnTransformer([
    ('std_scaler', StandardScaler(), num_cols),
    ('ord_encode', CustomOrdinalEncoder(), ordinal_cols),
    ('label_encode', OneHotEncoder(), cat_cols)], remainder= 'passthrough')

In [ ]:
data['Percentage'] = round((data['reading score'] + data['writing score']) / 2, 2)
data["Grade"] = data['Percentage'].apply(lambda x: Grade(x))

In [ ]:
X = data.drop('math score', axis = 1)
y = data['math score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= 42)

In [ ]:
X_train_prepared = pipeline.fit_transform(X_train)
X_test_prepared = pipeline.transform(X_test)

<div style='color: #216969;
           background-color: #EAF6F6;
           font-size: 200%;
           border-radius:15px;
           text-align:center;
           font-weight:600;
           border-style: solid;
           border-color: dark green;
           font-family: "Cambria";'>
Random Forest
<a class="anchor" id="6"></a> 

In [ ]:
model = RandomForestRegressor(random_state = 42)
model.fit(X_train_prepared, y_train)

kfold = KFold(n_splits= 5)
scores =  - cross_val_score(model, X_train_prepared, y_train, scoring="neg_mean_squared_error", cv=kfold)
rmse_scores = np.sqrt(scores)

print(f"Mean: {rmse_scores.mean()}", )
print(f"Standard deviation: {rmse_scores.std()}")

In [ ]:
y_pred = model.predict(X_test_prepared)
rmse = mean_squared_error(y_test, y_pred, squared= False)
r_square = r2_score(y_test, y_pred)

In [ ]:
print(f'Root Mean Squared error: {round(rmse, 3)}')
print(f'R-square: {round(r_square, 3)}')


In [ ]:
sns.set_context('notebook', font_scale= 1.3)
plt.figure(figsize= (10, 6))
sns.scatterplot(x= y_test, y= y_pred, color= '#005b96')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.show()

<div style='color: #216969;
           background-color: #EAF6F6;
           font-size: 200%;
           border-radius:15px;
           text-align:center;
           font-weight:600;
           border-style: solid;
           border-color: dark green;
           font-family: "Cambria";'>
Hyperparameter Optimization
<a class="anchor" id="7"></a> 

### **GridSearchCV**

In [ ]:
params = [
    {'n_estimators': [100, 250, 400], 'max_features': [8, 16, 24]},
    {'bootstrap': [False], 'n_estimators': [200, 400], 'max_features': [15, 30]},
  ]

model = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(model, params, cv = kfold,
                           scoring='neg_mean_squared_error',
                           return_train_score=True)

grid_search.fit(X_train_prepared, y_train)

In [ ]:
mean_rmse_grid_search = np.sqrt( - grid_search.best_score_)
print(f'RMSE: {round(mean_rmse_grid_search, 2)}')

### **RandomizedSearchCV**

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

params = {
        'n_estimators': randint(low= 200, high=500),
        'max_features': randint(low=10, high=50),
    }

model = RandomForestRegressor(random_state=42)
random_search = RandomizedSearchCV(model, params,
                                   n_iter = 10, cv = kfold, scoring='neg_mean_squared_error', random_state=42)

random_search.fit(X_train_prepared, y_train)

In [ ]:
mean_rmse_random_search = np.sqrt( - random_search.best_score_)
print(f'RMSE: {round(mean_rmse_random_search, 2)}')

### **Optuna**

In [ ]:
def random_forest_objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 600)
    max_depth = trial.suggest_int('max_depth', 5, 50)
    bootstrap = trial.suggest_categorical('bootstrap', ['True', 'False'])

    model = RandomForestRegressor(
        n_estimators= n_estimators,
        max_depth= max_depth,
        bootstrap= bootstrap
    )

    model.fit(X_train_prepared, y_train)
    cv_score = - cross_val_score(model, X_train_prepared, y_train, scoring= 'neg_mean_squared_error', cv= kfold)

    return np.sqrt(np.mean(cv_score))

study = optuna.create_study(direction= 'minimize')
study.optimize(random_forest_objective, n_trials= 10)

In [ ]:
trial = study.best_trial
print(f'RMSE: {round(trial.value, 2)}')

In [ ]:
params = {
    'n_estimators': 69,
    'max_depth': 21,
    'bootstrap': 'True'
}

<div style='color: #216969;
           background-color: #EAF6F6;
           font-size: 200%;
           border-radius:15px;
           text-align:center;
           font-weight:600;
           border-style: solid;
           border-color: dark green;
           font-family: "Cambria";'>
Final Predictions
<a class="anchor" id="8"></a> 

In [ ]:
final_model = RandomForestRegressor(**params)
final_model.fit(X_train_prepared, y_train)
final_predictions = final_model.predict(X_test_prepared)

final_rmse = mean_squared_error(y_test, final_predictions, squared= False)
final_rsquare = r2_score(y_test, final_predictions)

In [ ]:
print(f'RMSE: {round(final_rmse, 3)}')
print(f'R-square: {round(final_rsquare, 3)}')


In [ ]:
sns.set_context('notebook', font_scale= 1.3)
plt.figure(figsize= (10, 6))
sns.regplot(x= y_test, y= final_predictions, scatter_kws = {'s': 20, 'color': '#005b96', 'alpha': 0.7}, 
            line_kws = {'linewidth': 2, 'color': 'teal'})
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.show()

In [ ]:
# Distribution of error
sns.set_context('notebook', font_scale= 1.3)
plt.figure(figsize= (10, 6))
sns.histplot(y_test - final_predictions, color = 'teal', kde= True)
plt.xlabel('Residual');

In [ ]:
sns.set_context('notebook', font_scale= 1.3)
plt.figure(figsize= (10, 6))
sns.residplot(x = final_predictions, y = y_test, color = 'teal')
plt.xlabel('Predicted Maths Score')
plt.ylabel('Standardized Residuals')
plt.show()